# SymPy

NumPy and SciPy compute with numbers. SymPy computes with symbols, returning exact algebraic expressions. In quantum computing this means you can prove that $H^2 = I$ for all angles, find eigenvalues of parameterized Hamiltonians as closed-form expressions, and derive circuit identities before committing to numerical implementation.

This notebook accompanies **Lesson 10** of *Python Programming for Quantum Technology I*. Topics covered:
- Symbols, exact arithmetic, and simplification
- Symbolic matrices: parameterized gates, unitarity proofs, eigenvalues
- Dirac notation with `sympy.physics.quantum`
- Applying quantum gates symbolically with `qapply`
- `lambdify`: converting symbolic results to fast NumPy functions

In [ ]:
from sympy import (
    symbols, sqrt, pi, I, Rational,
    cos, sin, exp, conjugate,
    simplify, expand, trigsimp, expand_trig, factor,
    Matrix, eye, zeros,
    latex, init_printing, pprint
)
import numpy as np
import matplotlib.pyplot as plt

init_printing()   # render SymPy output as math in Jupyter

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

---
## 1. Symbols and Expressions

A **symbol** is a named placeholder that participates in exact algebraic computation. `symbols('x', real=True)` declares a real-valued symbol. Arithmetic on symbols produces new expressions, not numbers.

In [ ]:
theta, phi, t = symbols('theta phi t', real=True)

# Exact arithmetic: no floating-point approximation
print("sqrt(2)     =", sqrt(2))         # sqrt(2)
print("sqrt(2)**2  =", sqrt(2)**2)       # 2  (simplified exactly)
print("Rational    =", Rational(1, 3))   # 1/3
print("pi          =", pi)               # pi

In [ ]:
# Trigonometric identity: cos^2 + sin^2 = 1
expr = cos(theta)**2 + sin(theta)**2
print("Before simplify:", expr)
print("After trigsimp: ", trigsimp(expr))

print()

# Double-angle formula
expr2 = cos(2*theta)
print("expand_trig(cos(2θ)):", expand_trig(expr2))

In [ ]:
# Substitution: subs(symbol, value)
prob_0 = cos(theta / 2)**2

values = [0, pi/6, pi/4, pi/3, pi/2, pi]
for val in values:
    result = prob_0.subs(theta, val)
    print(f"  theta = {val}: P(|0>) = {result} = {float(result):.4f}")

In [ ]:
# LaTeX output for any expression
theta = symbols('theta')

rabi_formula = sin(theta / 2)**2

print("LaTeX:", latex(rabi_formula))
print()
print("Pretty print:")
pprint(rabi_formula)

### Exercise 1.1

The expectation value of $Z$ for the state $|\psi(\theta)\rangle = \cos(\theta/2)|0\rangle + \sin(\theta/2)|1\rangle$ is:

$$\langle Z \rangle(\theta) = \cos^2(\theta/2) - \sin^2(\theta/2)$$

1. Define this expression in SymPy using `theta = symbols('theta', real=True)`.
2. Use `trigsimp` to reduce it to its simplest form.
3. Evaluate it at $\theta = 0, \pi/2, \pi$ and verify the physical interpretation: $\langle Z \rangle = 1$ for $|0\rangle$, $0$ for $|+\rangle$, and $-1$ for $|1\rangle$.
4. Print the LaTeX representation of the simplified expression.

In [ ]:
from sympy import symbols, cos, sin, trigsimp, latex

theta = symbols('theta', real=True)

# Your code here


---
## 2. Symbolic Matrices

`sympy.Matrix` holds symbolic entries. Every matrix operation is exact. Multiply with `*`, compute the Hermitian conjugate with `.H`.

In [ ]:
s = 1 / sqrt(2)

X = Matrix([[0, 1], [1, 0]])
Y = Matrix([[0, -I], [I, 0]])
Z = Matrix([[1, 0], [0, -1]])
H_gate = Matrix([[s, s], [s, -s]])
S_gate = Matrix([[1, 0], [0, I]])
I2 = eye(2)   # 2x2 identity

print("Hadamard gate:")
pprint(H_gate)

In [ ]:
# Prove H^2 = I symbolically
H_squared = simplify(H_gate * H_gate)
print("H * H =")
pprint(H_squared)
print("H² = I:", H_squared == I2)

In [ ]:
# Gate conjugation identities: HXH = Z, HZH = X, HYH = -Y
results = {
    "HXH = Z":  simplify(H_gate * X * H_gate) == Z,
    "HZH = X":  simplify(H_gate * Z * H_gate) == X,
    "HYH = -Y": simplify(H_gate * Y * H_gate) == -Y,
    "X² = I":   simplify(X * X) == I2,
    "Y² = I":   simplify(Y * Y) == I2,
    "Z² = I":   simplify(Z * Z) == I2,
}

for name, result in results.items():
    print(f"  {name}: {result}")

### Parameterized Gates

In [ ]:
theta = symbols('theta', real=True)

Ry = Matrix([
    [cos(theta / 2), -sin(theta / 2)],
    [sin(theta / 2),  cos(theta / 2)]
])

print("R_Y(θ):")
pprint(Ry)

# Verify unitarity for ALL theta
unitarity_check = trigsimp(Ry.H * Ry)
print("\nR_Y†(θ) R_Y(θ) =")
pprint(unitarity_check)
print("Unitary for all θ:", unitarity_check == I2)

In [ ]:
# Prove rotation composition: R_Y(α) R_Y(β) = R_Y(α + β)
alpha, beta = symbols('alpha beta', real=True)

def Ry_sym(angle):
    return Matrix([
        [cos(angle / 2), -sin(angle / 2)],
        [sin(angle / 2),  cos(angle / 2)]
    ])

product   = trigsimp(Ry_sym(alpha) * Ry_sym(beta))
expected  = Ry_sym(alpha + beta).applyfunc(lambda e: trigsimp(expand_trig(e)))

diff = trigsimp(product - expected)
print("R_Y(α) R_Y(β) = R_Y(α+β):", diff == zeros(2, 2))

In [ ]:
# Symbolic eigenvalues of the transverse-field Ising Hamiltonian
# For a single bond: H = -J*ZZ - h*(XI + IX) gives a 4x4 matrix
J_sym, h_sym = symbols('J h', positive=True)

ZZ = Matrix([[ 1, 0, 0, 0],
             [ 0,-1, 0, 0],
             [ 0, 0,-1, 0],
             [ 0, 0, 0, 1]])
XI = Matrix([[0, 0, 1, 0],
             [0, 0, 0, 1],
             [1, 0, 0, 0],
             [0, 1, 0, 0]])
IX = Matrix([[0, 1, 0, 0],
             [1, 0, 0, 0],
             [0, 0, 0, 1],
             [0, 0, 1, 0]])

H_ising = -J_sym * ZZ - h_sym * (XI + IX)

print("Computing eigenvalues... (may take a moment)")
evals = H_ising.eigenvals()
print("Eigenvalues:")
for val, mult in evals.items():
    print(f"  {simplify(val)}  (multiplicity {mult})")

### Exercise 2.1

**Proving the $T$ gate is the square root of $S$.**

The $T$ gate is defined as $T = \begin{pmatrix}1 & 0 \\ 0 & e^{i\pi/4}\end{pmatrix}$ and the $S$ gate as $S = \begin{pmatrix}1 & 0 \\ 0 & i\end{pmatrix}$.

1. Define both gates as SymPy matrices.
2. Compute $T^2$ and verify it equals $S$.
3. Compute $T^4$ and verify it equals $Z$.
4. Compute $T^8$ and verify it equals the identity.
5. Print the results using `simplify`.

In [ ]:
from sympy import Matrix, I, pi, exp, simplify, eye

T = Matrix([[1, 0], [0, exp(I * pi / 4)]])
S = Matrix([[1, 0], [0, I]])
Z = Matrix([[1, 0], [0, -1]])
I2 = eye(2)

# Your code here


---
## 3. Dirac Notation

`sympy.physics.quantum` provides abstract Dirac notation as symbolic objects. `Ket` and `Bra` are abstract labels; `Qubit` is a concrete computational basis state with gate semantics.

In [ ]:
from sympy.physics.quantum import Ket, Bra, Dagger, InnerProduct, OuterProduct, TensorProduct
from sympy import sqrt

ket_0 = Ket('0')
ket_1 = Ket('1')
bra_0 = Bra('0')

print("|0> =", ket_0)
print("<0| =", Dagger(ket_0))
print("Dagger(|0>) == <0|:", Dagger(ket_0) == bra_0)

# Inner product <0|0>
print("<0|0> =", InnerProduct(bra_0, ket_0))

# Outer product |0><0| (projector onto |0>)
print("|0><0| =", OuterProduct(ket_0, bra_0))

In [ ]:
# Superposition states
ket_plus  = (Ket('0') + Ket('1')) / sqrt(2)
ket_minus = (Ket('0') - Ket('1')) / sqrt(2)

print("|+> =", ket_plus)
print("|->, =", ket_minus)

# Two-qubit tensor product
ket_00 = TensorProduct(Ket('0'), Ket('0'))
ket_11 = TensorProduct(Ket('1'), Ket('1'))

bell_plus = (ket_00 + ket_11) / sqrt(2)
print("\n|Φ+> =", bell_plus)

### Applying Quantum Gates with `qapply`

In [ ]:
from sympy.physics.quantum.qubit import Qubit, qubit_to_matrix
from sympy.physics.quantum.gate import HadamardGate, XGate, ZGate, CNOT
from sympy.physics.quantum.qapply import qapply

# Single-qubit operations
q0 = Qubit('0')
q1 = Qubit('1')

H_on_0 = HadamardGate(0)

result_plus  = qapply(H_on_0 * q0)
result_minus = qapply(H_on_0 * q1)

print("H|0> =", result_plus)
print("H|1> =", result_minus)

In [ ]:
# Derive the Bell state |Φ+> symbolically: H on q1, then CNOT(1, 0)
# SymPy numbers qubits right-to-left: Qubit('ab') has q1=a, q0=b

initial = Qubit('00')
print("Initial state:", initial)

# Step 1: Apply Hadamard to qubit 1 (the left bit)
after_H = qapply(HadamardGate(1) * initial)
print("After H on q1:", after_H)

# Step 2: Apply CNOT (control = 1, target = 0)
after_CNOT = qapply(CNOT(1, 0) * after_H)
print("After CNOT:   ", after_CNOT)

In [ ]:
# Convert the Bell state to a column matrix and then to NumPy
bell_matrix = qubit_to_matrix(after_CNOT)
print("Bell state as column vector:")
pprint(bell_matrix)

# Convert to NumPy
bell_np = np.array(bell_matrix.tolist(), dtype=complex).flatten()
print("\nNumPy array:", bell_np)
print("Probabilities:", np.abs(bell_np)**2)

### Exercise 3.1

**Deriving the GHZ state symbolically.**

The three-qubit GHZ state is:

$$|\text{GHZ}\rangle = \frac{1}{\sqrt{2}}\left(|000\rangle + |111\rangle\right)$$

It is prepared by:
1. Applying $H$ to qubit 2.
2. Applying CNOT with control=2, target=1.
3. Applying CNOT with control=1, target=0.

Using `Qubit`, `HadamardGate`, `CNOT`, and `qapply`:
1. Derive the state after each step symbolically.
2. Print the state at each step.
3. Convert the final state to a NumPy column vector using `qubit_to_matrix`.
4. Verify that only $|000\rangle$ and $|111\rangle$ have nonzero probability.

In [ ]:
from sympy.physics.quantum.qubit import Qubit, qubit_to_matrix
from sympy.physics.quantum.gate import HadamardGate, CNOT
from sympy.physics.quantum.qapply import qapply
import numpy as np

# Your code here


---
## 4. From Symbolic to Numerical: `lambdify`

`lambdify` converts a SymPy expression into a Python function backed by NumPy. This is the bridge from symbolic derivation to fast numerical evaluation.

In [ ]:
from sympy import symbols, cos, sin, Matrix, lambdify
import numpy as np

theta = symbols('theta', real=True)

# Symbolic R_Y gate
Ry_sym = Matrix([
    [cos(theta / 2), -sin(theta / 2)],
    [sin(theta / 2),  cos(theta / 2)]
])

# Convert to a NumPy function
Ry_func = lambdify(theta, Ry_sym, modules='numpy')

print("R_Y(0):",     np.round(Ry_func(0), 6))
print("R_Y(pi/2):",  np.round(Ry_func(np.pi / 2), 6))
print("R_Y(pi):",    np.round(Ry_func(np.pi), 6))   # should give X (approx)

In [ ]:
# Derive and plot Bloch sphere probabilities
theta = symbols('theta', real=True)

prob_0_sym = cos(theta / 2)**2
prob_1_sym = sin(theta / 2)**2

print("P(|0>):", trigsimp(prob_0_sym))
print("P(|1>):", trigsimp(prob_1_sym))
print("P(|0>) + P(|1>) = 1:", trigsimp(prob_0_sym + prob_1_sym) == 1)

p0_func = lambdify(theta, prob_0_sym, modules='numpy')
p1_func = lambdify(theta, prob_1_sym, modules='numpy')

thetas = np.linspace(0, 2 * np.pi, 400)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(thetas, p0_func(thetas), color="steelblue", linewidth=2.5, label=r"$P(|0\rangle) = \cos^2(\theta/2)$")
ax.plot(thetas, p1_func(thetas), color="tomato",    linewidth=2.5, label=r"$P(|1\rangle) = \sin^2(\theta/2)$",
        linestyle="--")
ax.set_xlabel(r"$\theta$ (radians)", fontsize=12)
ax.set_ylabel("Probability", fontsize=12)
ax.set_title("Bloch Sphere Probabilities (Symbolic Formula, Numerical Plot)", fontsize=12)
ax.set_xticks([0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi])
ax.set_xticklabels(["0", "π/2", "π", "3π/2", "2π"])
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Multi-variable lambdify: full Bloch sphere state
theta_s, phi_s = symbols('theta phi', real=True)

alpha_sym = cos(theta_s / 2)
beta_sym  = exp(I * phi_s) * sin(theta_s / 2)

state_func = lambdify([theta_s, phi_s], [alpha_sym, beta_sym], modules='numpy')

# Verify normalization at a random point
test_theta, test_phi = 1.2, 0.7
a0, a1 = state_func(test_theta, test_phi)

print(f"theta={test_theta}, phi={test_phi}")
print(f"alpha = {a0:.4f}")
print(f"beta  = {a1:.4f}")
print(f"|alpha|^2 + |beta|^2 = {abs(a0)**2 + abs(a1)**2:.8f}")

### Exercise 4.1

**Symbolic derivation of the Rabi formula, then numerical simulation.**

The state after applying $R_Y(\theta)$ to $|0\rangle$ is:

$$|\psi(\theta)\rangle = \cos(\theta/2)|0\rangle + \sin(\theta/2)|1\rangle$$

1. Derive $P(|1\rangle) = \sin^2(\theta/2)$ symbolically.
2. Use `trigsimp` to simplify it to an equivalent form using $1 - \cos\theta$.
3. Derive the expression for $\langle X \rangle = \langle\psi|X|\psi\rangle$ symbolically and simplify it.
4. Use `lambdify` to convert both $P(|1\rangle)$ and $\langle X \rangle$ to NumPy functions.
5. Plot both on the same axes over $\theta \in [0, 2\pi]$. Label the axes and add a legend.

In [ ]:
from sympy import symbols, cos, sin, Matrix, lambdify, trigsimp, simplify
import numpy as np
import matplotlib.pyplot as plt

theta = symbols('theta', real=True)

X = Matrix([[0, 1], [1, 0]])

# Your code here


---
## Summary

| Task | SymPy tool | Example |
|------|-----------|--------|
| Declare variables | `symbols('x', real=True)` | `theta = symbols('theta', real=True)` |
| Exact fractions / roots | `Rational`, `sqrt` | `sqrt(2)**2 == 2` |
| Simplify | `simplify`, `trigsimp` | `trigsimp(cos(x)**2 + sin(x)**2) == 1` |
| Substitute | `.subs(sym, val)` | `expr.subs(theta, pi/2)` |
| LaTeX output | `latex(expr)` | `latex(cos(theta/2)**2)` |
| Symbolic matrix | `Matrix([[...]])` | gate matrices with symbols |
| Hermitian conjugate | `.H` | `Ry.H * Ry == eye(2)` |
| Eigenvalues | `.eigenvals()` | ground state energy as expression in $J, h$ |
| Dirac notation | `Ket`, `Bra`, `Dagger` | abstract state manipulation |
| Gate application | `qapply(Gate * Qubit)` | step-by-step circuit derivation |
| To NumPy | `lambdify(sym, expr, 'numpy')` | evaluate symbolic formula at many points |

**What comes next:** This lesson completes *Python Programming for Quantum Technology I*. You now have the full scientific Python toolkit: Python basics, NumPy arrays, Matplotlib visualization, SciPy algorithms, and SymPy symbolic computation. The next course, *Python Programming for Quantum Technology II*, applies all of this to quantum circuits, state simulation, noise modeling, and variational algorithms.

---
## Challenge Problem

**Symbolic quantum Fourier transform on 2 qubits.**

The 2-qubit quantum Fourier transform (QFT) circuit applies:
1. $H$ to qubit 1.
2. A controlled-$S$ gate (control=1, target=0): applies $S$ to qubit 0 when qubit 1 is $|1\rangle$. Use `CGate(1, SGate(0))` from `sympy.physics.quantum.gate`.
3. $H$ to qubit 0.
4. A SWAP of qubits 0 and 1.

The 4x4 QFT matrix (in the computational basis $|00\rangle, |01\rangle, |10\rangle, |11\rangle$) is:

$$\text{QFT}_4 = \frac{1}{2}\begin{pmatrix}1 & 1 & 1 & 1 \\ 1 & i & -1 & -i \\ 1 & -1 & 1 & -1 \\ 1 & -i & -1 & i\end{pmatrix}$$

1. Apply the circuit step by step to each of the four basis states $|00\rangle, |01\rangle, |10\rangle, |11\rangle$ using `qapply`.
2. Convert each output to a column vector with `qubit_to_matrix`.
3. Assemble the four columns into a 4x4 SymPy matrix (each input basis state produces one column of the QFT matrix).
4. Compare to the known $\text{QFT}_4$ matrix above using `simplify`.
5. Verify that the assembled matrix is unitary.

In [ ]:
from sympy.physics.quantum.qubit import Qubit, qubit_to_matrix
from sympy.physics.quantum.gate import HadamardGate, SwapGate, CGate, PhaseGate
from sympy.physics.quantum.qapply import qapply
from sympy import Matrix, I, sqrt, simplify, eye, Rational

# The S gate is PhaseGate(0) in SymPy (applies e^(i*pi/2) = i to |1>)
# Controlled-S: CGate(1, PhaseGate(0)) means control=qubit 1, target=PhaseGate on qubit 0

def apply_qft_circuit(qubit_state):
    """Apply the 2-qubit QFT circuit to a Qubit state."""
    # Your code here
    pass

basis_states = [Qubit('00'), Qubit('01'), Qubit('10'), Qubit('11')]

# Your code here: build the QFT matrix column by column
